**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 Session 9: vLLM 로컬 서빙 환경 구축

## 📋 학습 목표

1️⃣ vLLM의 핵심 기술(PagedAttention, Continuous Batching)을 이해한다  
2️⃣ vLLM Python API로 오프라인 추론을 수행한다  
3️⃣ vLLM OpenAI 호환 API 서버를 로컬에서 구축한다  

---

### 🖥️ 실습 환경

| 환경 | GPU | 실습 모델 | 비고 |
|------|-----|----------|------|
| **강사 서버** | TITAN RTX 24GB ×2 | Qwen2.5-7B-Instruct | 멀티GPU 시연 가능 |
| **학생 PC** | RTX 4060 8GB | Qwen2.5-1.5B-Instruct | vLLM 로컬 실습 가능 |

> 📌 이 노트북은 **로컬 GPU에서 직접 실행**하는 것을 목표로 합니다.  
> RTX 4060(8GB)에서도 1.5B 모델로 vLLM 실습이 가능합니다.

## 1️⃣ vLLM 소개

**vLLM**은 대규모 언어 모델을 빠르게 서빙하기 위한 오픈소스 엔진입니다.

### 🚀 핵심 기술

- 🔹 **PagedAttention**: GPU 메모리를 페이지 단위로 관리하여 KV Cache 메모리 낭비를 90% 이상 줄임
- 🔹 **Continuous Batching**: 요청이 완료될 때마다 새 요청을 즉시 슬롯에 삽입 → 대기 시간 최소화
- 🔹 **KV Cache 최적화**: 키-밸류 캐시를 비연속 메모리 블록에 저장하여 GPU 메모리 활용률 극대화
- 🔹 **OpenAI 호환 API**: `openai` 라이브러리로 바로 호출 가능 → 기존 코드 그대로 재사용

### 📊 Static vs Continuous Batching

```
Static Batching (기존 방식):
  요청1: [████████████████████]  → 모든 요청이 끝날 때까지 대기
  요청2: [████████____대기____]  → 짧은 응답도 긴 응답 끝날 때까지 대기
  요청3: [██████████████______]  → GPU 활용률 낮음

Continuous Batching (vLLM):
  요청1: [████████████████████]  → 끝나면 즉시 새 요청 투입
  요청2: [████████] → 요청4: [████████████]  → 빈 슬롯 재활용
  요청3: [██████████████] → 요청5: [██████]  → GPU 항상 바쁨
```

In [1]:
# 📊 vLLM vs Ollama vs Transformers 서빙 비교
print("📊 LLM 서빙 프레임워크 비교")
print("=" * 70)

comparisons = [
    ("설치 난이도",    "보통",          "매우 쉬움",       "매우 쉬움"),
    ("처리량(QPS)",   "높음 (10x+)",   "보통 (3x)",      "기준 (1x)"),
    ("동시 요청",     "수백 개",        "수십 개",         "1개"),
    ("배칭",          "연속 배칭",      "단일 배칭",       "없음"),
    ("모델 포맷",     "HF (FP16/AWQ)", "GGUF (양자화)",   "HF (모든 형식)"),
    ("API 호환",      "OpenAI",        "OpenAI",         "없음"),
    ("최소 VRAM",     "6GB+ (1.5B)",   "4GB+ (1.5B)",    "2GB+"),
    ("주요 용도",     "프로덕션 서빙",  "개발/테스트",     "실험/학습"),
]

print(f"{'항목':<15} {'vLLM':>17} {'Ollama':>17} {'Transformers':>17}")
print("-" * 70)
for item, vllm, ollama, tf in comparisons:
    print(f"{item:<15} {vllm:>17} {ollama:>17} {tf:>17}")

print(f"\n💡 선택 가이드:")
print(f"  🔹 프로덕션 서빙 (다수 동시 사용자) → vLLM")
print(f"  🔹 로컬 개발/테스트 → Ollama")
print(f"  🔹 연구/커스텀 추론 → Transformers")

📊 LLM 서빙 프레임워크 비교
항목                           vLLM            Ollama      Transformers
----------------------------------------------------------------------
설치 난이도                         보통             매우 쉬움             매우 쉬움
처리량(QPS)                높음 (10x+)           보통 (3x)           기준 (1x)
동시 요청                        수백 개              수십 개                1개
배칭                          연속 배칭             단일 배칭                없음
모델 포맷               HF (FP16/AWQ)        GGUF (양자화)        HF (모든 형식)
API 호환                     OpenAI            OpenAI                없음
최소 VRAM               6GB+ (1.5B)       4GB+ (1.5B)              2GB+
주요 용도                     프로덕션 서빙            개발/테스트             실험/학습

💡 선택 가이드:
  🔹 프로덕션 서빙 (다수 동시 사용자) → vLLM
  🔹 로컬 개발/테스트 → Ollama
  🔹 연구/커스텀 추론 → Transformers


## 2️⃣ vLLM 설치 & 환경 확인

vLLM은 pip으로 설치할 수 있습니다. CUDA 12.1 이상이 필요합니다.

```bash
# 터미널에서 설치
pip install vllm
```

> ⚠️ vLLM은 설치 시간이 오래 걸릴 수 있습니다 (5~10분).  
> 설치가 안 되는 경우, CUDA 버전과 Python 버전(3.9+)을 확인하세요.

In [2]:
# 🔧 vLLM 설치 확인 + GPU 환경 점검
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # GPU1 사용 (GPU0이 다른 프로세스에 사용될 수 있음)

import torch
import sys

print("🔧 환경 점검")
print("=" * 50)

# Python 버전
print(f"Python: {sys.version.split()[0]}")

# PyTorch & CUDA
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f}GB")
    
    # GPU별 안내
    if vram_gb >= 20:
        print(f"\n✅ VRAM {vram_gb:.0f}GB → 7B 모델 FP16 서빙 가능")
        default_model = "Qwen/Qwen2.5-7B-Instruct"
    elif vram_gb >= 6:
        print(f"\n✅ VRAM {vram_gb:.0f}GB → 1.5B 모델 FP16 서빙 가능")
        default_model = "Qwen/Qwen2.5-1.5B-Instruct"
    else:
        print(f"\n⚠️ VRAM {vram_gb:.0f}GB → vLLM 사용이 어렵습니다")
        default_model = None
    
    if default_model:
        print(f"📌 기본 실습 모델: {default_model}")
else:
    print("⚠️ GPU를 사용할 수 없습니다.")
    default_model = None

# vLLM 설치 확인
print()
try:
    import vllm
    print(f"✅ vLLM: {vllm.__version__}")
    vllm_available = True
except ImportError:
    print("❌ vLLM이 설치되어 있지 않습니다.")
    print("   터미널에서 실행: pip install vllm")
    vllm_available = False

🔧 환경 점검
Python: 3.12.3
PyTorch: 2.10.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA TITAN RTX
VRAM: 23.5GB

✅ VRAM 23GB → 7B 모델 FP16 서빙 가능
📌 기본 실습 모델: Qwen/Qwen2.5-7B-Instruct

✅ vLLM: 0.17.1


## 3️⃣ vLLM 사용 방식: 오프라인 추론 vs API 서버

vLLM은 **두 가지 방식**으로 사용할 수 있습니다. 목적에 따라 선택합니다.

---

### 방식 A: 오프라인 추론 (Python API)

**Python 코드 안에서 직접** 모델을 로드하고 추론합니다. 서버가 필요 없습니다.

```python
from vllm import LLM, SamplingParams

# 1) 모델 로딩 (GPU 메모리에 올림)
llm = LLM(model="Qwen/Qwen2.5-7B-Instruct", dtype="float16")  # 모델을 GPU에 로딩

# 2) 프롬프트 준비 & 추론
outputs = llm.generate(["한국의 수도는?", "파이썬이란?"], SamplingParams(max_tokens=128))  # 바로 추론

# 3) 결과 확인
for out in outputs:
    print(out.outputs[0].text)  # 결과 확인
```

**특징:**
- 코드가 간단하고 직관적
- 배치 처리에 유리 (여러 프롬프트를 리스트로 한 번에 전달)
- 스크립트/노트북에서 실험할 때 적합
- 단일 프로세스에서만 사용 가능 (외부에서 호출 불가)

---

### 방식 B: API 서버 (OpenAI 호환)

**터미널에서 서버를 먼저 띄우고**, 코드에서는 HTTP API로 호출합니다.

```bash
# 터미널에서 서버 실행 (모델 로딩까지 수 분 소요)
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct \
    --dtype float16 --port 9100
```

```python
# 코드에서 OpenAI 라이브러리로 호출 (기존 OpenAI 코드와 동일!)
from openai import OpenAI

client = OpenAI(base_url="http://localhost:9100/v1", api_key="not-needed")
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": "한국의 수도는?"}],
)
print(response.choices[0].message.content)
```

**특징:**
- OpenAI API와 **완전히 동일한 인터페이스** → 기존 코드 그대로 재사용
- Continuous Batching으로 **동시 요청을 자동으로 효율 처리**
- 여러 클라이언트(노트북, 앱, curl 등)가 **동시에 접속 가능**
- 프로덕션 서빙, 웹앱 백엔드에 적합

---

### 비교 요약

```
┌────────────────────────────┬─────────────────────────────────────┐
│   방식 A: 오프라인 추론     │   방식 B: API 서버                  │
├────────────────────────────┼─────────────────────────────────────┤
│ 서버 필요: ❌               │ 서버 필요: ✅ (터미널에서 실행)      │
│ 호출 방법: llm.generate()  │ 호출 방법: client.chat.completions  │
│ 동시 요청: ❌ (단일 프로세스)│ 동시 요청: ✅ (수백 개 동시 처리)   │
│ 용도: 실험, 배치 처리       │ 용도: 프로덕션, 다중 클라이언트     │
│ 장점: 코드가 간단           │ 장점: 높은 처리량, OpenAI 호환      │
├────────────────────────────┴─────────────────────────────────────┤
│ ⚠️ 같은 GPU에서 동시에 실행할 수 없음!                           │
│    → 방식 A 실습 후 커널 재시작 → 방식 B 실습                    │
└──────────────────────────────────────────────────────────────────┘
```

> 💡 **현업에서는?**  
> 실험/개발 단계에서는 **방식 A**로 빠르게 테스트하고,  
> 서비스 배포 시에는 **방식 B**로 API 서버를 운영합니다.
>
> 이 노트북에서는 **방식 A → 방식 B** 순서로 실습합니다.

In [ ]:
# 🚀 vLLM 오프라인 추론 - 모델 로딩
import time

if not vllm_available:
    print("⚠️ vLLM이 설치되어 있지 않아 이 셀을 건너뜁니다.")
    print("   pip install vllm 으로 설치 후 다시 실행하세요.")
elif not torch.cuda.is_available():
    print("⚠️ GPU를 사용할 수 없어 이 셀을 건너뜁니다.")
else:
    from vllm import LLM, SamplingParams
    
    # GPU VRAM에 따른 모델 선택
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram_gb >= 20:
        model_name = "Qwen/Qwen2.5-7B-Instruct"
        max_len = 2048
        mem_util = 0.85
    else:
        model_name = "Qwen/Qwen2.5-1.5B-Instruct"  # 학생용은 HF 이름 유지
        max_len = 1024
        mem_util = 0.80
    
    print(f"🚀 모델 로딩: {model_name}")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   max_model_len={max_len}, gpu_memory_utilization={mem_util}")
    
    start_load = time.time()
    llm = LLM(
        model=model_name,
        dtype="float16",
        max_model_len=max_len,
        gpu_memory_utilization=mem_util,
        enforce_eager=True,  # TITAN RTX(Turing) Flash Attention 2 미지원 → eager 모드 필수
    )
    load_time = time.time() - start_load
    print(f"✅ 모델 로딩 완료 ({load_time:.1f}초)")
    
    # 샘플링 파라미터
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=128,
    )
    
    # 단일 프롬프트 추론
    prompts = [
        "인공지능의 미래에 대해 간단히 설명해주세요.",
    ]
    
    print(f"\n📝 단일 프롬프트 추론")
    start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - start
    
    for output in outputs:
        generated = output.outputs[0].text
        num_tokens = len(output.outputs[0].token_ids)
        print(f"\n❓ {output.prompt}")
        print(f"💬 {generated}")
        print(f"\n📊 {num_tokens}개 토큰 / {elapsed:.2f}초 = {num_tokens/elapsed:.1f} tok/s")

🚀 모델 로딩: Qwen/Qwen2.5-7B-Instruct
   GPU: NVIDIA TITAN RTX
   max_model_len=2048, gpu_memory_utilization=0.85
INFO 04-01 10:35:39 [utils.py:238] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}


INFO 04-01 10:35:41 [model.py:531] Resolved architecture: Qwen2ForCausalLM
WARNING 04-01 10:35:41 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 04-01 10:35:41 [model.py:1554] Using max model len 2048
INFO 04-01 10:35:41 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-01 10:35:41 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-01 10:35:41 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-01 10:35:41 [vllm.py:957] Cudagraph is disabled under eager mode


/home/ejkim/multicampus/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


(EngineCore_DP0 pid=868363) INFO 04-01 10:35:48 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_

(EngineCore_DP0 pid=868363) Process EngineCore_DP0:
(EngineCore_DP0 pid=868363) Traceback (most recent call last):
(EngineCore_DP0 pid=868363)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=868363)     self.run()
(EngineCore_DP0 pid=868363)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=868363)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=868363)   File "/home/ejkim/multicampus/venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1104, in run_engine_core
(EngineCore_DP0 pid=868363)     raise e
(EngineCore_DP0 pid=868363)   File "/home/ejkim/multicampus/venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1090, in run_engine_core
(EngineCore_DP0 pid=868363)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore_DP0 pid=868363)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore_DP0 pid

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
# vLLM 로딩 후 재사용
# 단일 프롬프트 추론
prompts = [
    "AGI는 가능할까?.",
]

print(f"\n📝 단일 프롬프트 추론")
start = time.time()
outputs = llm.generate(prompts, sampling_params)
elapsed = time.time() - start

for output in outputs:
    generated = output.outputs[0].text
    num_tokens = len(output.outputs[0].token_ids)
    print(f"\n❓ {output.prompt}")
    print(f"💬 {generated}")
    print(f"\n📊 {num_tokens}개 토큰 / {elapsed:.2f}초 = {num_tokens/elapsed:.1f} tok/s")


📝 단일 프롬프트 추론


Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.22s/it, est. speed input: 2.18 toks/s, output: 39.82 toks/s]


❓ AGI는 가능할까?.
💬  AI에 대한 논의가 계속되고 있습니다. 그러나 전문가들은 그 동안 AGI(인지적 지능)의 가능성에 대해 논의해 왔습니다. 그들은 대부분 AGI가 가능한 것으로 생각하고 있습니다. 그러나 이것이 반드시 좋은 일인지, 혹은 우리가 준비되어 있는지에 대해서는 계속해서 논의가 필요합니다. 그렇다면 AGI에 대한 전문가들의 의견은 어떻게 되는지 알려주실 수 있나요?
AGI(인지적 지능)에 대한 전문가들의 의견은 다양하며, 그 가능성에 대한 태도는 크게

📊 128개 토큰 / 3.23초 = 39.6 tok/s


In [ ]:
# 📊 배칭 성능 확인 - 1개 vs 여러 프롬프트 동시 처리

if not vllm_available or not torch.cuda.is_available():
    print("⚠️ vLLM/GPU를 사용할 수 없어 이 셀을 건너뜁니다.")
else:
    try:
        llm  # 이전 셀에서 로드한 모델 재사용
    except NameError:
        print("⚠️ 이전 셀에서 모델을 먼저 로드해주세요.")
    else:
        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.9,
            max_tokens=128,
        )
        
        # 테스트 프롬프트 5개
        batch_prompts = [
            "파이썬의 장점을 3가지 알려주세요.",
            "머신러닝과 딥러닝의 차이점은?",
            "한국의 대표적인 전통 음식 3가지는?",
            "클라우드 컴퓨팅이란 무엇인가요?",
            "좋은 코드를 작성하는 방법은?",
        ]
        
        # 1) 개별 추론 (1개씩 5번)
        print("📊 배칭 성능 비교")
        print("=" * 50)
        
        start = time.time()
        individual_tokens = 0
        for p in batch_prompts:
            out = llm.generate([p], sampling_params)
            individual_tokens += len(out[0].outputs[0].token_ids)
        individual_time = time.time() - start
        
        # 2) 배치 추론 (5개 한 번에)
        start = time.time()
        batch_outputs = llm.generate(batch_prompts, sampling_params)
        batch_time = time.time() - start
        batch_tokens = sum(len(o.outputs[0].token_ids) for o in batch_outputs)
        
        # 결과 비교
        print(f"\n🔸 개별 추론 (1개씩 × 5회):")
        print(f"   시간: {individual_time:.2f}초, 토큰: {individual_tokens}개, {individual_tokens/individual_time:.1f} tok/s")
        
        print(f"\n🔹 배치 추론 (5개 동시):")
        print(f"   시간: {batch_time:.2f}초, 토큰: {batch_tokens}개, {batch_tokens/batch_time:.1f} tok/s")
        
        speedup = individual_time / batch_time if batch_time > 0 else 0
        print(f"\n🚀 배치 처리 속도 향상: {speedup:.1f}x")
        print(f"\n💡 vLLM의 연속 배칭은 여러 프롬프트를 동시에 처리하여")
        print(f"   GPU 활용률을 높이고 전체 처리량을 크게 증가시킵니다.")

In [ ]:
# 🧹 GPU 메모리 정리
import gc

try:
    del llm
    print("✅ vLLM 모델 해제")
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"✅ GPU 메모리 정리 완료 (현재 사용: {allocated:.1f}GB)")

## 4️⃣ vLLM OpenAI 호환 API 서버

vLLM의 진짜 강점은 **API 서버**입니다.  
서버를 띄우면 `openai` 라이브러리로 바로 호출할 수 있고,  
Continuous Batching 덕분에 **동시 요청**을 효율적으로 처리합니다.

### 🔄 서버 방식의 장점
- 모델을 한 번만 로드하고 여러 클라이언트가 공유
- 동시 요청 자동 배칭 → 높은 처리량
- OpenAI 호환 → 기존 코드/라이브러리 그대로 사용

---

### ⚠️ 아래 셀 실행 전에 반드시 3단계를 순서대로 수행하세요!

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│  Step 1. 노트북 커널 재시작                                      │
│          → 메뉴: Kernel → Restart Kernel                        │
│          → 오프라인 추론에서 사용한 GPU 메모리를 해제합니다       │
│                                                                 │
│  Step 2. 터미널에서 vLLM API 서버 실행                           │
│          → 아래 명령어를 터미널에 복사해서 실행하세요             │
│                                                                 │
│  Step 3. "Uvicorn running on ..." 메시지가 나오면                │
│          → 아래 코드 셀 실행                                     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 📋 Step 2 서버 실행 명령어

**TITAN RTX (24GB) / VRAM 20GB 이상:**
```bash
CUDA_VISIBLE_DEVICES=1 /home/ejkim/LLM_Advanced/venv/bin/python \
    -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct \
    --dtype float16 \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.85 \
    --enforce-eager \
    --port 9100
```

**RTX 4060 (8GB):**
```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype float16 \
    --max-model-len 1024 \
    --gpu-memory-utilization 0.80 \
    --enforce-eager \
    --port 9100
```

> 📌 서버는 모델을 GPU에 올리므로 **시작까지 수 분**이 걸립니다.  
> `INFO: Uvicorn running on http://0.0.0.0:9000` 메시지가 나오면 준비 완료입니다.

In [ ]:
# 🌐 vLLM API 서버 실행 명령어 (터미널에서 실행)
import torch

print("🌐 vLLM OpenAI 호환 API 서버 실행 방법")
print("=" * 60)
print("\n⚠️ 아래 명령어는 '터미널'에서 실행하세요 (노트북 X)")
print("   서버가 실행된 상태에서 이 노트북의 다음 셀들을 실행합니다.")

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
else:
    vram_gb = 0
    gpu_name = "N/A"

print(f"\n현재 GPU: {gpu_name} ({vram_gb:.0f}GB)")

if vram_gb >= 20:
    print(f"""
━━━ 🟢 VRAM 20GB+ 권장 명령어 ━━━

python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-7B-Instruct \\
    --dtype float16 \\
    --max-model-len 4096 \\
    --gpu-memory-utilization 0.85 \\
    --port 9100
""")
else:
    print(f"""
━━━ 🟡 VRAM 8GB (RTX 4060) 권장 명령어 ━━━

python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-1.5B-Instruct \\
    --dtype float16 \\
    --max-model-len 1024 \\
    --gpu-memory-utilization 0.80 \\
    --port 9100
""")

print("서버가 'INFO: Uvicorn running on ...' 메시지를 출력하면 준비 완료!")

In [ ]:
# ⚙️ vLLM 주요 서버 파라미터 정리
params = [
    ("--model",                   "HuggingFace 모델 이름",             "필수"),
    ("--dtype",                   "데이터 타입 (float16/bfloat16/auto)", "auto"),
    ("--max-model-len",           "최대 시퀀스 길이",                   "모델 기본값"),
    ("--gpu-memory-utilization",  "GPU 메모리 사용 비율 (0~1)",        "0.9"),
    ("--tensor-parallel-size",    "텐서 병렬 GPU 수",                  "1"),
    ("--max-num-seqs",            "동시 처리 최대 시퀀스 수",           "256"),
    ("--quantization",            "양자화 (awq/gptq/squeezellm)",     "없음"),
    ("--port",                    "API 서버 포트",                      "8000"),
    ("--host",                    "바인딩 호스트",                      "0.0.0.0"),
    ("--served-model-name",       "API에서 사용할 모델 별칭",           "--model과 동일"),
]

print("⚙️ vLLM API 서버 주요 파라미터")
print("=" * 75)
print(f"{'파라미터':<30} {'설명':<28} {'기본값':>14}")
print("-" * 75)
for param, desc, default in params:
    print(f"{param:<30} {desc:<28} {default:>14}")

In [ ]:
# 📡 Python OpenAI 클라이언트로 vLLM 서버 호출
# ⚠️ 이 셀을 실행하기 전에:
#   1) 이전 셀의 오프라인 추론 커널을 재시작 (GPU 메모리 해제)
#   2) 터미널에서 vLLM 서버를 실행하세요 (위 셀 참고)

import requests
import time

VLLM_BASE_URL = "http://localhost:9100/v1"

# 서버 상태 확인 (최대 5회 재시도)
vllm_server_running = False
print("🔍 vLLM 서버 연결 확인 중...")

for attempt in range(5):
    try:
        resp = requests.get("http://localhost:9100/health", timeout=5)
        if resp.status_code == 200:
            print("✅ vLLM 서버가 실행 중입니다!")
            vllm_server_running = True
            break
    except requests.ConnectionError:
        pass
    
    if attempt < 4:
        print(f"   재시도 {attempt+1}/5... (5초 대기)")
        time.sleep(5)

if not vllm_server_running:
    print("❌ vLLM 서버에 연결할 수 없습니다.")
    print("\n🔧 터미널에서 아래 명령어를 실행하세요:")
    print("""
    CUDA_VISIBLE_DEVICES=1 python -m vllm.entrypoints.openai.api_server \\
        --model Qwen/Qwen2.5-7B-Instruct \\
        --dtype float16 \\
        --max-model-len 2048 \\
        --gpu-memory-utilization 0.85 \\
        --enforce-eager \\
        --port 9100
    """)
    print("   서버가 'Uvicorn running on ...' 메시지를 출력하면 이 셀을 다시 실행하세요.")
else:
    from openai import OpenAI
    
    client = OpenAI(
        base_url=VLLM_BASE_URL,
        api_key="not-needed",
    )
    
    # 사용 가능한 모델 확인
    models = client.models.list()
    model_id = models.data[0].id
    print(f"📌 서빙 중인 모델: {model_id}")
    
    # Chat Completions API 호출
    start = time.time()
    
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다."},
            {"role": "user", "content": "파이썬의 장점을 3가지 알려주세요."},
        ],
        temperature=0.7,
        max_tokens=256,
    )
    
    elapsed = time.time() - start
    
    print(f"\n💬 응답:\n{response.choices[0].message.content}")
    print(f"\n📊 응답 시간: {elapsed:.2f}초")
    print(f"   토큰 사용: 입력 {response.usage.prompt_tokens}, 출력 {response.usage.completion_tokens}")

In [ ]:
# 🌊 스트리밍 응답 테스트

if not vllm_server_running:
    print("⚠️ vLLM 서버가 실행 중이 아닙니다. 위 셀을 먼저 확인하세요.")
else:
    print("🌊 스트리밍 응답 테스트")
    print("=" * 50)
    print("❓ 질문: 딥러닝이 무엇인지 초보자에게 설명해주세요.")
    print("\n💬 응답: ", end="", flush=True)
    
    start = time.time()
    token_count = 0
    
    stream = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": "딥러닝이 무엇인지 초보자에게 설명해주세요."}],
        temperature=0.7,
        max_tokens=200,
        stream=True,
    )
    
    first_token_time = None
    for chunk in stream:
        if chunk.choices[0].delta.content:
            if first_token_time is None:
                first_token_time = time.time() - start
            print(chunk.choices[0].delta.content, end="", flush=True)
            token_count += 1
    
    total_time = time.time() - start
    print(f"\n\n📊 성능:")
    print(f"   첫 토큰 시간 (TTFT): {first_token_time:.3f}초")
    print(f"   총 시간: {total_time:.2f}초")
    print(f"   토큰 수: {token_count}개")
    print(f"   처리 속도: {token_count/total_time:.1f} tok/s")

In [3]:
# 🔄 인터랙티브 체감 비교: Transformers(직접 추론) vs vLLM(API 서버) 스트리밍 속도
#
# ⚠️ 실행 전 필수 조건:
#   1) 커널 재시작 (Kernel → Restart Kernel) — 이전 GPU 메모리 해제
#   2) 터미널에서 vLLM 서버 실행 중 (포트 9000, GPU 1)
#   3) 이 셀을 가장 먼저 실행하세요 (CUDA_VISIBLE_DEVICES 설정 필요)
#
# 💡 듀얼 GPU 환경 전용 (GPU 0: Transformers, GPU 1: vLLM 서버)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 반드시 torch import 전에! (GPU 0 = Transformers용)

import time
import torch
import gc
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread
from openai import OpenAI

# ── 모델 선택 (VRAM 기반) ──
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb >= 20:
    model_name = "Qwen/Qwen2.5-7B-Instruct"
else:
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# ── Transformers 모델 로드 ──
print(f"📦 Transformers 모델 로딩: {model_name}")
print(f"   GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f}GB)")
print(f"   (최초 실행 시 1~2분 소요)\n")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)
vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"✅ Transformers 로딩 완료! (GPU 메모리: {vram_used:.1f}GB 사용 중)")

# ── vLLM 서버 확인 ──
vllm_ok = False
try:
    resp = requests.get("http://localhost:9100/health", timeout=3)
    if resp.status_code == 200:
        vllm_client = OpenAI(base_url="http://localhost:9100/v1", api_key="not-needed")
        vllm_model_id = vllm_client.models.list().data[0].id
        vllm_ok = True
        print(f"✅ vLLM 서버 연결: {vllm_model_id} (포트 9000)")
except Exception:
    pass

if not vllm_ok:
    print("❌ vLLM 서버(포트 9000)에 연결할 수 없습니다!")
    print("   터미널에서 vLLM 서버를 먼저 실행하세요.")
else:
    print(f"\n{'='*60}")
    print("💬 질문을 입력하면 Transformers → vLLM 순서로 추론 후 비교합니다")
    print("   종료하려면 'q' 입력 또는 빈 입력 후 Enter")
    print(f"{'='*60}")

    while True:
        query = input("\n❓ 질문: ").strip()
        if not query or query.lower() in ('q', 'quit', 'exit'):
            break

        messages = [
            {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요."},
            {"role": "user", "content": query},
        ]

        # ────────────────────────────────────────
        # 🐢 Step 1: Transformers 스트리밍
        # ────────────────────────────────────────
        print(f"\n{'─'*60}")
        print(f"🐢 [1/2] Transformers (GPU 직접 추론, 최적화 없음)")
        print(f"{'─'*60}")

        tf_error = None
        tf_tokens = 0
        tf_first = None
        tf_total = 0
        tf_tps = 0

        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors="pt").to(tf_model.device)
            streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

            gen_kwargs = {
                "input_ids": inputs["input_ids"],
                "attention_mask": inputs["attention_mask"],
                "max_new_tokens": 200,
                "do_sample": True,
                "temperature": 0.7,
                "top_p": 0.9,
                "streamer": streamer,
            }

            tf_start = time.time()
            thread = Thread(target=tf_model.generate, kwargs=gen_kwargs)
            thread.start()

            for tok_text in streamer:
                if tok_text:
                    if tf_first is None:
                        tf_first = time.time() - tf_start
                    print(tok_text, end="", flush=True)
                    tf_tokens += 1
            thread.join()
            tf_total = time.time() - tf_start
            tf_tps = tf_tokens / tf_total if tf_total > 0 else 0
            if tf_first is None:
                tf_first = tf_total

            print(f"\n⏱️  첫 토큰: {tf_first:.3f}초 | 총: {tf_total:.2f}초 | {tf_tps:.1f} tok/s")

        except Exception as e:
            tf_error = str(e)
            print(f"\n⚠️ Transformers 추론 에러: {e}")

        # GPU 캐시 정리 (다음 추론 안정성)
        torch.cuda.empty_cache()

        # ────────────────────────────────────────
        # 🚀 Step 2: vLLM 스트리밍
        # ────────────────────────────────────────
        print(f"\n{'─'*60}")
        print(f"🚀 [2/2] vLLM (API 서버, PagedAttention + Continuous Batching)")
        print(f"{'─'*60}")

        v_start = time.time()
        v_tokens = 0
        v_first = None

        stream = vllm_client.chat.completions.create(
            model=vllm_model_id,
            messages=messages,
            max_tokens=200,
            temperature=0.7,
            stream=True,
        )

        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                if v_first is None:
                    v_first = time.time() - v_start
                print(delta, end="", flush=True)
                v_tokens += 1

        v_total = time.time() - v_start
        v_tps = v_tokens / v_total if v_total > 0 else 0
        if v_first is None:
            v_first = v_total

        print(f"\n⏱️  첫 토큰: {v_first:.3f}초 | 총: {v_total:.2f}초 | {v_tps:.1f} tok/s")

        # ────────────────────────────────────────
        # 📊 Step 3: 비교 결과 정리
        # ────────────────────────────────────────
        print(f"\n{'━'*60}")
        print(f"📊 비교 결과")
        print(f"{'━'*60}")

        if tf_error:
            print(f"  ⚠️ Transformers 에러로 비교 불가: {tf_error}")
        else:
            print(f"  {'항목':<15} {'🐢 Transformers':>18} {'🚀 vLLM':>18}")
            print(f"  {'─'*51}")
            print(f"  {'첫 토큰(TTFT)':<15} {tf_first:>17.3f}초 {v_first:>17.3f}초")
            print(f"  {'총 소요 시간':<15} {tf_total:>17.2f}초 {v_total:>17.2f}초")
            print(f"  {'생성 토큰 수':<15} {tf_tokens:>17}개 {v_tokens:>17}개")
            print(f"  {'처리 속도':<15} {tf_tps:>16.1f}t/s {v_tps:>16.1f}t/s")

            speedup = tf_total / v_total if v_total > 0 else 0
            if speedup > 1:
                print(f"\n  🏆 vLLM이 약 {speedup:.1f}배 빠릅니다!")
                print(f"     → PagedAttention + 최적화된 CUDA 커널의 효과")
            elif speedup > 0:
                print(f"\n  📌 Transformers가 약 {1/speedup:.1f}배 빠릅니다 (단일 요청에서는 비슷할 수 있음)")
            print(f"  {'─'*51}")

# ── 메모리 정리 ──
print("\n🧹 Transformers 모델 메모리 해제 중...")
try:
    del tf_model, tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("✅ 완료")


📦 Transformers 모델 로딩: /home/ejkim/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28
   GPU: NVIDIA TITAN RTX (23GB)
   (최초 실행 시 1~2분 소요)



Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.17s/it]


✅ Transformers 로딩 완료! (GPU 메모리: 14.2GB 사용 중)
✅ vLLM 서버 연결: Qwen/Qwen2.5-7B-Instruct (포트 9000)

💬 질문을 입력하면 스트리밍 속도 차이를 눈으로 체감할 수 있습니다
   종료하려면 'q' 입력 또는 빈 입력 후 Enter

────────────────────────────────────────────────────────────
🐢 Transformers (GPU 직접 추론, 최적화 없음)
────────────────────────────────────────────────────────────
AGI (인간 수준의 인공지능, Artificial General Intelligence)에 대한 구체적인 도래 시기는 아직 불확실합니다. 많은 연구자들과 전문가들은 AGI가 가능한 미래에 존재할 것으로 예측하지만, 그 시기는 아직 정확히 예측하기 어렵습니다.

현재의 AI 기술은 특정 작업이나 분야에서 인간을 능가하는 성능을 보여주고 있지만, 인간과 같은 유연성과 창의성, 다양한 지식의 통합을 가지고 일반적인 문제를 해결하는 능력은 아직 보유하고 있지 않습니다.

기술 발전 속도와 연구 투자의 증가에 따라 AGI 도래 시기는 가까워질 것으로 전문가들은 예상하고 있습니다. 그러나 이는 여전히 복잡한 과제로 남아 있으며, 다양한 연구진 사이에서도 다양한 의견이 있습니다.

결론적으로
⏱️  첫 토큰: 0.693초 | 총: 10.01초 | 8.4 tok/s

────────────────────────────────────────────────────────────
🚀 vLLM (API 서버, PagedAttention + Continuous Batching)
────────────────────────────────────────────────────────────
현재로서는 AGI (인간 수준 인공지능)의 정확한 도래 시기는 불명확합니다. 과학자들과 연구자들은 AGI가 가능한 기술적인 가능성에

## 6️⃣ GPU별 vLLM 추천 설정

GPU VRAM에 따라 서빙 가능한 모델과 권장 설정이 달라집니다.

In [ ]:
# 📋 GPU별 vLLM 모델/설정 가이드
print("📋 GPU별 vLLM 추천 설정")
print("=" * 75)

gpu_configs = [
    {
        "gpu": "RTX 4060 (8GB)",
        "models": [
            ("Qwen2.5-1.5B-Instruct", "float16", 1024, 0.80),
        ],
        "note": "1.5B까지 가능, 3B 이상은 VRAM 부족",
    },
    {
        "gpu": "RTX 4090 (24GB)",
        "models": [
            ("Qwen2.5-7B-Instruct", "float16", 4096, 0.85),
            ("Qwen2.5-7B-Instruct-AWQ", "auto", 8192, 0.85),
        ],
        "note": "7B FP16 또는 7B AWQ 양자화",
    },
    {
        "gpu": "TITAN RTX (24GB)",
        "models": [
            ("Qwen2.5-7B-Instruct", "float16", 2048, 0.85),
        ],
        "note": "Turing 아키텍처, 7B FP16 가능 (max_model_len 제한 권장)",
    },
    {
        "gpu": "TITAN RTX ×2 (48GB)",
        "models": [
            ("Qwen2.5-14B-Instruct", "float16", 4096, 0.85),
        ],
        "note": "tensor-parallel-size=2로 멀티GPU 서빙",
    },
    {
        "gpu": "A100 (40GB/80GB)",
        "models": [
            ("Qwen2.5-14B-Instruct", "float16", 8192, 0.90),
            ("Qwen2.5-72B-Instruct-AWQ", "auto", 4096, 0.90),
        ],
        "note": "프로덕션 권장 GPU",
    },
]

for config in gpu_configs:
    print(f"\n🔹 {config['gpu']}")
    print(f"   {config['note']}")
    for model, dtype, max_len, mem in config["models"]:
        print(f"   → {model} (dtype={dtype}, max_len={max_len}, mem={mem})")

# 현재 GPU에 맞는 추천
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"\n" + "=" * 75)
    print(f"📌 현재 GPU: {gpu_name} ({vram_gb:.0f}GB)")
    if vram_gb >= 20:
        print(f"   → 7B 모델 FP16 서빙을 추천합니다.")
    elif vram_gb >= 6:
        print(f"   → 1.5B 모델 FP16 서빙을 추천합니다.")
    else:
        print(f"   → vLLM 대신 Ollama 사용을 추천합니다.")

In [ ]:
# 📌 핵심 요약
print("📌 Session 9 핵심 정리")
print("=" * 50)
print()
print("1️⃣  vLLM은 PagedAttention + Continuous Batching으로")
print("    높은 처리량의 LLM 서빙을 제공합니다.")
print()
print("2️⃣  Python API (오프라인 추론):")
print("    → LLM() 로 모델 로드, generate()로 배치 추론")
print("    → 여러 프롬프트를 한 번에 처리하면 처리량 크게 증가")
print()
print("3️⃣  API 서버:")
print("    → python -m vllm.entrypoints.openai.api_server")
print("    → OpenAI 호환 → 기존 openai 라이브러리 그대로 사용")
print()
print("4️⃣  GPU별 가이드:")
print("    → RTX 4060 (8GB): 1.5B 모델까지")
print("    → TITAN RTX (24GB): 7B 모델 FP16")
print("    → A100 (40GB+): 14B~72B 모델")


---

## 📎 부록: GPU 클라우드 서비스 참고

로컬 GPU로 부족한 경우 (7B+ 모델, 멀티GPU 실험 등), 아래 클라우드 서비스를 활용할 수 있습니다.

| 서비스 | GPU | 무료 | 가격대 | 특징 |
|--------|-----|------|--------|------|
| **Google Colab** | T4 (무료) / A100 (Pro) | T4 15GB 무료 | $10/월 (Pro) | 주피터 환경, 구글 드라이브 연동 |
| **Kaggle** | T4 x2 / P100 | 30시간/주 무료 | 무료 | 데이터셋 풍부, 경진대회 |
| **RunPod** | A100 / H100 | ❌ | $0.44/시간~ | 다양한 GPU, vLLM 템플릿 |
| **Lambda** | A100 / H100 | ❌ | $0.50/시간~ | ML 특화, 빠른 셋업 |
| **Vast.ai** | 다양 | ❌ | $0.10/시간~ | 저렴, 커뮤니티 GPU |

### 💡 클라우드 GPU 선택 팁
- **학습/실습**: Colab Free 또는 Kaggle (무료)
- **파인튜닝**: RunPod A100 40GB ($0.44/시간)
- **서빙**: RunPod H100 (안정적) 또는 Vast.ai (저렴)
- **비용 절감**: Spot/Preemptible 인스턴스 활용

---

## 🎯 실습 과제

1️⃣ vLLM Python API로 Qwen2.5-1.5B-Instruct 모델의 오프라인 추론을 실행해보세요  
2️⃣ vLLM API 서버를 실행하고, Python OpenAI 클라이언트로 호출해보세요  
3️⃣ Ollama와 vLLM에 같은 프롬프트 5개를 보내서 응답 시간을 비교해보세요  
4️⃣ (심화) 동시 요청 수를 1, 5, 10으로 변경하며 vLLM의 처리량 변화를 관찰해보세요  

---

## 📚 참고 자료
- [vLLM 공식 문서](https://docs.vllm.ai/)
- [vLLM GitHub](https://github.com/vllm-project/vllm)
- [PagedAttention 논문](https://arxiv.org/abs/2309.06180)
- [vLLM 블로그](https://blog.vllm.ai/)